## Sustainability & Spatial Misalignment in Early-Stage Corridor Development

| Hypothesis | Statement |
|---|---|
| H1 | Built-up expansion is associated with declining vegetation cover and increased surface heat susceptibility |
| H2 | A share of new built-up areas overlaps with environmentally sensitive or water-prone zones |
| H3 |  Areas with low economic utilisation disproportionately coincide with higher environmental exposure |

In [ ]:
import geemap
import ee 
import geemap.colormaps as cm
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

Map = geemap.Map()

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')
Map.centerObject(roi, 10)
Map

In [ ]:
# ── Sentinel-2: 2025 (Oct–Dec post-monsoon composite) ───────
s2_2025 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

# ── Sentinel-2: 2016 (Oct–Dec post-monsoon composite) ───────
s2_2016 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2016-10-01', '2016-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

Computes NDBI, MNDWI, SAVI, and NDVI for a given Sentinel-2 image.  
NDVI is added here (not in RQ1/RQ2) as it provides a cleaner vegetation-loss signal alongside SAVI.

In [ ]:
def calculate_indices(image):
    """
    Computes NDBI, MNDWI, SAVI, NDVI from a Sentinel-2 SR image.
    Band references:
        B2=Blue, B3=Green, B4=Red, B8=NIR, B11=SWIR1
    """
    ndbi = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': image.select('B11'), 'NIR': image.select('B8')}
    ).rename('NDBI')

    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)',
        {'Green': image.select('B3'), 'SWIR1': image.select('B11')}
    ).rename('MNDWI')

    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)',
        {'NIR': image.select('B8'), 'Red': image.select('B4')}
    ).rename('SAVI')

    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    return image.addBands([ndbi, mndwi, savi, ndvi])


processed_2025 = calculate_indices(s2_2025)
processed_2016 = calculate_indices(s2_2016)

print("Spectral indices computed for 2016 and 2025.")

---
## Built-up Masks (Reusing RQ1 thresholds)

| Year | NDBI threshold | Rationale |
|---|---|---|
| 2025 | > 0.05 | Post-monsoon; lower saline soil noise |
| 2016 | > 0.13 | Higher radiometric contrast from saline pans in drier 2016 season |

In [ ]:
# ── 2025 Built-up Mask ───────────────────────────────────────
built_up_2025 = processed_2025.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2025.select('NDBI'),
        'MNDWI': processed_2025.select('MNDWI'),
        'SAVI':  processed_2025.select('SAVI')
    }
).rename('BuiltUp_2025')

# ── 2016 Built-up Mask ───────────────────────────────────────
built_up_2016 = processed_2016.expression(
    '(NDBI > 0.13) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2016.select('NDBI'),
        'MNDWI': processed_2016.select('MNDWI'),
        'SAVI':  processed_2016.select('SAVI')
    }
).rename('BuiltUp_2016')

print("Built-up masks ready: 2016 and 2025.")

---
### OUTPUT 1: Land Transformation Map

A two-panel view of:
- **Built-up change** (2016 → 2025) — reusing the 4-class growth heatmap from RQ1
- **Vegetation loss** — NDVI/SAVI change between 2016 and 2025

Both layers are shown together so you can read **where built-up growth co-occurred with vegetation loss**.

Built-up Growth Heatmap (2016 → 2025)

| Value | Class | Colour |
|---|---|---|
| 0 | No built-up (both years) | ⬛ |
| 1 | Stable built-up | ⬜ |
| 2 | New growth 2016→2025 | 🟠 |
| 3 | Lost built-up | 🔵 |

In [ ]:
# ── Built-up Change Classification ──────────────────────────
change_stack = built_up_2016.rename('y2016').addBands(built_up_2025.rename('y2025'))

growth_class = change_stack.expression(
    "(b('y2016') == 0 && b('y2025') == 0) ? 0"   # No built-up
    ": (b('y2016') == 1 && b('y2025') == 1) ? 1"  # Stable
    ": (b('y2016') == 0 && b('y2025') == 1) ? 2"  # New Growth
    ": (b('y2016') == 1 && b('y2025') == 0) ? 3"  # Lost
    ": 0"
).rename('GrowthClass').clip(roi)

### Vegetation Loss Layer (NDVI change & SAVI change)

Negative delta = vegetation lost.  
We compute both NDVI delta and SAVI delta - SAVI is more reliable over Dholera's arid/saline background.

In [ ]:
# ── Vegetation Change Layers ─────────────────────────────────
# NDVI delta: negative = vegetation loss
ndvi_2025 = processed_2025.select('NDVI')
ndvi_2016 = processed_2016.select('NDVI')
ndvi_delta = ndvi_2025.subtract(ndvi_2016).rename('NDVI_Delta').clip(roi)

# SAVI delta: more soil-noise-robust over arid landscapes
savi_2025 = processed_2025.select('SAVI')
savi_2016 = processed_2016.select('SAVI')
savi_delta = savi_2025.subtract(savi_2016).rename('SAVI_Delta').clip(roi)

print("Vegetation change layers computed.")

In [ ]:

# --- Map 3: SAVI/NDBI (Testing)---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
test_params = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}

Map_Test.addLayer(savi_2025, test_params, 'SAVI (2025)')
Map_Test.addLayer(savi_2016, test_params, 'SAVI (2016)')
Map_Test.addLayer(ndvi_2025, test_params, 'NDVI (2025)')
Map_Test.addLayer(ndvi_2016, test_params, 'NDVI (2016)')
Map_Test.addLayer(savi_delta, test_params, 'SAVI (Delta)')
Map_Test.addLayer(ndvi_delta, test_params, 'NDVI (Delta)')
Map_Test.add_colorbar(vis_params=test_params, label="SAVI Value", orientation="horizontal")
print("Displaying Test Map...")
display(Map_Test)

#### Visualise: Land Transformation Map

In [ ]:
# ── Map: Land Transformation ─────────────────────────────────
Map_LT = geemap.Map()
Map_LT.centerObject(roi, 11)

# --- Reference: True Colour 2025 ---
Map_LT.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# --- Layer 1: Built-up Growth Heatmap ---
Map_LT.addLayer(
    growth_class,
    {'min': 0, 'max': 3, 'palette': ['1a1a1a', '888888', 'ff4500', '1e90ff']},
    '🔥 Built-up Growth (2016→2025)'
)

# --- Layer 2: NDVI Delta ---
# Red = loss, Green = gain
Map_LT.addLayer(
    ndvi_delta,
    {'min': -0.4, 'max': 0.4, 'palette': ['d73027', 'f7f7f7', '1a9850']},
    'NDVI Change (2016→2025)', shown=False
)

# --- Layer 3: SAVI Delta ---
# More reliable over Dholera's salt-pan background
Map_LT.addLayer(
    savi_delta,
    {'min': -0.3, 'max': 0.3, 'palette': ['8b0000', 'f5f5f5', '006400']},
    'SAVI Change (2016→2025) ← Primary Vegetation Layer'
)

# --- Legend ---
Map_LT.add_legend(
    title='Built-up Change 2016–2025',
    legend_dict={
        'No Built-up (Both Years)':      '1a1a1a',
        'Stable Built-up':               '888888',
        '🟠 New Growth (2025 Only)':     'ff4500',
        '🔵 Lost Built-up (2016 Only)':  '1e90ff',
    },
    position='bottomleft'
)

# ROI boundary
Map_LT.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Land Transformation Map ready — toggle layers in the panel.")
display(Map_LT)

### Vegetation Loss Area Statistics

Measures the area where SAVI *declined* (vegetation lost) within the new growth footprint - directly tests **H1**.

In [ ]:
# ── Area Stats: Vegetation Loss ──────────────────────────────
pixel_area = ee.Image.pixelArea()

# Mask: pixels where SAVI declined (vegetation lost)
veg_loss_mask = savi_delta.lt(0).rename('VegLoss')

# Mask: new growth pixels only (class 2 from growth_class)
new_growth_pixels = growth_class.eq(2)

# Intersection: new growth AND vegetation declined
new_growth_veg_loss = new_growth_pixels.And(veg_loss_mask)

def area_km2(mask, band_name='BuiltUp'):
    result = mask.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    )
    return round(result.getInfo().get(band_name, 0) / 1e6, 3)

total_new_growth_km2 = area_km2(new_growth_pixels.rename('BuiltUp'), 'BuiltUp')
veg_loss_total_km2   = area_km2(veg_loss_mask.rename('BuiltUp'), 'BuiltUp')
growth_veg_loss_km2  = area_km2(new_growth_veg_loss.rename('BuiltUp'), 'BuiltUp')

print("=" * 55)
print("  VEGETATION CHANGE REPORT (2016 → 2025)")
print("=" * 55)
print(f"  🟠 Total New Built-up Growth       : {total_new_growth_km2:>8.3f} km²")
print(f"  🌿 Total SAVI Decline (whole ROI)  : {veg_loss_total_km2:>8.3f} km²")
print(f"  ⚠️  New Growth ∩ Vegetation Loss   : {growth_veg_loss_km2:>8.3f} km²")
if total_new_growth_km2 > 0:
    pct = round(growth_veg_loss_km2 / total_new_growth_km2 * 100, 1)
    print(f"  → {pct}% of new built-up growth occurred on previously vegetated land.")
print("=" * 55)